# Production-Ready Investor–Startup Recommender
This notebook rebuilds the recommender from scratch using a single, end-to-end scikit-learn pipeline. The API scores **investor profiles against all startups** and returns the best-matching startups for each investor's preferences. All preprocessing and modeling steps are encapsulated in one saved `.pkl` file so inference only needs raw inputs.

## Feature schema used
Each training row represents one (startup, investor) pair with raw inputs plus a few engineered flags that are computed **inside** the pipeline.

**Inference direction:** one fixed **investor profile** is scored against every startup; the top-ranked startups are returned.

**Raw inputs (passed at inference):**
- `startup_category`, `startup_budget_required`, `startup_stage`, `startup_location`
- `investor_category`, `investor_budget_range`, `investor_preferred_stage`, `investor_location`
- (optional) `startup_risk_level`, `startup_traction_level`, `investor_risk_preference`, `investor_traction_preference`

**Engineered in-pipeline (no manual preprocessing):**
- `category_match_flag`
- `location_match_flag`
- `stage_match_flag` (startup stage in investor preferred stages)
- `budget_rank_diff` and `budget_close_flag` (ordinal proximity of budget ranges)

In [ ]:
from __future__ import annotations

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from recommender import (
    PairFeatureBuilder,
    RAW_FEATURES,
    make_rule_based_matches,
    normalize_investors,
    normalize_startups,
    recommend,
)


In [ ]:
DATA_DIR = Path('.')
MATCHES_PATH = DATA_DIR / 'recommended_investors.csv'
STARTUPS_PATH = DATA_DIR / 'firebase_startup_profiles.csv'
INVESTORS_PATH = DATA_DIR / 'firebase_investor_profiles.csv'

if not STARTUPS_PATH.exists():
    STARTUPS_PATH = DATA_DIR / 'startup2.csv'
if not INVESTORS_PATH.exists():
    INVESTORS_PATH = DATA_DIR / 'investor2.csv'

startups_raw = pd.read_csv(STARTUPS_PATH)
investors_raw = pd.read_csv(INVESTORS_PATH)

if MATCHES_PATH.exists():
    matches = pd.read_csv(MATCHES_PATH)
    if 'label' not in matches.columns:
        matches['label'] = 1
    is_rule_based = (
        'source' in matches.columns
        and matches['source'].astype(str).str.contains('rule_based', case=False).any()
    )
    if is_rule_based:
        matches = make_rule_based_matches(startups_raw, investors_raw, top_k=5)
        matches['source'] = 'rule_based_investor_preferences_from_available_profiles'
        print(f'Regenerated {len(matches)} investor-preference matches from available profiles.')
else:
    matches = make_rule_based_matches(startups_raw, investors_raw, top_k=5)
    matches['source'] = 'rule_based_investor_preferences_from_available_profiles'
    print(f'Generated {len(matches)} rule-based positive matches because {MATCHES_PATH.name} was not found.')

startups = normalize_startups(startups_raw)
investors = normalize_investors(investors_raw)

print(f'Loaded {len(startups)} startups from {STARTUPS_PATH.name}')
print(f'Loaded {len(investors)} investors from {INVESTORS_PATH.name}')
startups.head()


In [ ]:
def build_pair_dataset(
    matches_df: pd.DataFrame,
    startups_df: pd.DataFrame,
    investors_df: pd.DataFrame,
    neg_ratio: float = 1.0,
    seed: int = 42,
    max_negatives_per_investor: int | None = None,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    all_startups = startups_df['startup_id'].unique()
    negative_rows: list[dict] = []

    for investor_id, group in matches_df.groupby('investor_id'):
        positive_startups = set(group['startup_id'].tolist())
        candidates = np.setdiff1d(all_startups, list(positive_startups))
        n_pos = len(group)
        n_neg = int(np.ceil(n_pos * neg_ratio))
        if max_negatives_per_investor is not None:
            n_neg = min(n_neg, max_negatives_per_investor)
        n_neg = min(n_neg, len(candidates))
        if n_neg <= 0:
            continue
        sampled = rng.choice(candidates, size=n_neg, replace=False)
        for startup_id in sampled:
            negative_rows.append(
                {'startup_id': startup_id, 'investor_id': investor_id, 'label': 0}
            )

    negatives = pd.DataFrame(negative_rows)
    pairs = pd.concat([matches_df[['startup_id', 'investor_id', 'label']], negatives], ignore_index=True)

    dataset = (
        pairs.merge(startups_df, on='startup_id', how='left')
        .merge(investors_df, on='investor_id', how='left')
    )
    return dataset

pair_dataset = build_pair_dataset(matches, startups, investors, neg_ratio=1.0, max_negatives_per_investor=10)
pair_dataset.head()

In [ ]:
# PairFeatureBuilder lives in recommender.py so saved models can be loaded
# from notebooks, scripts, and the FastAPI app without __main__ pickle errors.
PairFeatureBuilder


In [ ]:
raw_features = RAW_FEATURES

categorical_features = [
    'startup_category',
    'startup_budget_required',
    'startup_stage',
    'startup_location',
    'startup_risk_level',
    'startup_traction_level',
    'investor_category',
    'investor_budget_range',
    'investor_preferred_stage',
    'investor_location',
    'investor_risk_preference',
    'investor_traction_preference',
    'category_match_flag',
    'location_match_flag',
    'stage_match_flag',
    'budget_close_flag',
]

numeric_features = ['budget_rank_diff']

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('numeric', StandardScaler(), numeric_features),
    ],
    remainder='drop',
)

model = LogisticRegression(
    max_iter=500,
    class_weight='balanced',
    n_jobs=None,
)

pipeline = Pipeline(
    steps=[
        ('feature_builder', PairFeatureBuilder()),
        ('preprocess', preprocessor),
        ('model', model),
    ]
)

X = pair_dataset[raw_features].fillna('')
y = pair_dataset['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipeline.fit(X_train, y_train)

proba = pipeline.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)

print('ROC-AUC:', roc_auc_score(y_test, proba))
print(classification_report(y_test, preds))


In [ ]:
MODEL_PATH = DATA_DIR / "startup_investor_pipeline.pkl"
joblib.dump(pipeline, MODEL_PATH)
MODEL_PATH

In [ ]:
example_investor = {
    'category': 'FinTech',
    'budget_range': '500k-2M',
    'preferred_stage': 'Growth,Scaling',
    'location': 'Pakistan',
    'risk_preference': 'Medium',
    'traction_preference': 'Revenue',
}

recommend(example_investor, startups_raw, pipeline, top_n=40)


## API testing (after saving the model)

Run from the project folder in PowerShell.

**This computer only (local CSV mode):**
```powershell
cd C:\Users\moham\OneDrive\Documents\model
Remove-Item Env:USE_FIRESTORE -ErrorAction SilentlyContinue
.\.venv\Scripts\python.exe -m uvicorn main:app --host 127.0.0.1 --port 8000
```

**Firestore mode on this computer:**
```powershell
cd C:\Users\moham\OneDrive\Documents\model
$env:USE_FIRESTORE = "1"
.\.venv\Scripts\python.exe -m uvicorn main:app --host 127.0.0.1 --port 8000
```

**Same Wi-Fi / LAN testing from phone or another device:**
```powershell
cd C:\Users\moham\OneDrive\Documents\model
$env:USE_FIRESTORE = "1"
.\.venv\Scripts\python.exe -m uvicorn main:app --host 0.0.0.0 --port 8000
```

Use `--host 0.0.0.0` (not `127.0.0.1`) so other devices on your network can reach the API.

Find your LAN IP:
```powershell
Get-NetIPAddress -AddressFamily IPv4 | Where-Object { $_.InterfaceAlias -notlike '*Loopback*' -and $_.IPAddress -notlike '169.*' } | Select-Object -ExpandProperty IPAddress
```

Then open on this PC:
- Index: `http://127.0.0.1:8000/`
- Health: `http://127.0.0.1:8000/health`
- Swagger docs: `http://127.0.0.1:8000/docs`

On another device on the same Wi-Fi (replace with your LAN IP):
- Health: `http://192.168.18.207:8000/health`
- Docs: `http://192.168.18.207:8000/docs`
- Recommend: `POST http://192.168.18.207:8000/recommend`

Example request body in `/docs`:
```json
{
  "category": "FinTech",
  "budget_range": "500k-2M",
  "preferred_stage": "Growth,Scaling",
  "location": "Pakistan",
  "risk_preference": "Medium",
  "traction_preference": "Revenue"
}
```